# 05 · Reporting Views

**Purpose:** Summarize predictive vs standardized skill views using model outputs.

**Inputs:** reports/fg_success_calibration.csv, reports/aipw_calibration.csv

**Outputs:** reports/reporting_predictive_view.csv, reports/reporting_standardized_view.csv

- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers-≤40-lines-each)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [ ]:
# Parameters & Modes
SMOKE_MODE <- TRUE
FULL_MODE <- !SMOKE_MODE

reference_dir <- 'Reference'
data_dir <- 'data'
reports_dir <- 'reports'
config_path <- file.path('config', 'params.yaml')

if (!dir.exists(reports_dir)) {
  dir.create(reports_dir, recursive = TRUE)
}

params <- list(
  time_knots = c(60, 120, 300),
  p_clip_min = 0.05,
  p_clip_max = 0.95,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  distance_cap = 65,
  yardline_spline_df = 5,
  weight_floor = 0.1,
  weight_cap = 10,
  late_game_threshold = 120,
  distance_spline_df = 6,
  wind_bins = c(0, 5, 10, 15, 25),
  default_p_hat = 0.5,
  default_m_hat = 0.65,
  overall_success_rate = 0.85
)

if (file.exists(config_path)) {
  tryCatch({
    config_params <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_params, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

list2env(params, envir = .GlobalEnv)
set.seed(101)


In [ ]:
# Imports — install if missing
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'scales'
)

install_if_missing <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
  }
}

invisible(purrr::walk(dependencies, install_if_missing))

library(dplyr)
library(tibble)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(ggplot2)
library(mgcv)
library(splines)
library(glmmTMB)
library(pROC)
library(yaml)
library(scales)


### Function Index
- `get_schema()` — quick schema glance at data frames.
- `add_time_features()` — derive score/time convenience features.
- `ensure_columns()` — add fallback columns with default values.
- `clip_weights()` — enforce weight floor/cap.
- `stabilize_weights()` — compute stabilized attempt weights.
- `calc_brier()` — calculate (weighted) Brier score.
- `plot_calibration()` — convenience calibration scatter/smoother.


In [ ]:
# Utilities & Helpers (≤40 lines each)
get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    example = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

add_time_features <- function(df) {
  df %>%
    mutate(
      score_diff = dplyr::coalesce(score_diff, score_differential, 0),
      time_remaining = dplyr::coalesce(game_seconds_remaining, quarter_seconds_remaining, 0),
      log_time_remaining = log1p(time_remaining),
      late_game = time_remaining <= late_game_threshold,
      one_score = abs(score_diff) <= 8
    )
}

ensure_columns <- function(df, defaults) {
  for (nm in names(defaults)) {
    if (!nm %in% names(df)) {
      df[[nm]] <- defaults[[nm]]
    }
  }
  df
}

clip_weights <- function(w, floor = weight_floor, cap = weight_cap) {
  pmin(pmax(w, floor), cap)
}

stabilize_weights <- function(p_hat, base_rate) {
  clip_weights(base_rate / p_hat)
}

calc_brier <- function(actual, predicted, weights = NULL) {
  if (is.null(weights)) {
    mean((predicted - actual) ^ 2)
  } else {
    sum(weights * (predicted - actual) ^ 2) / sum(weights)
  }
}

plot_calibration <- function(df, prob_col, outcome_col, group_col, path) {
  plot <- ggplot(df, aes_string(x = prob_col, y = outcome_col, color = group_col)) +
    geom_point(alpha = 0.4) +
    geom_smooth(method = 'loess', se = FALSE) +
    labs(title = 'Calibration', x = 'Predicted', y = 'Observed')
  ggsave(path, plot = plot, width = 6, height = 4, dpi = 150)
  invisible(plot)
}


In [ ]:
# Data Load & Peek
fg_calibration_path <- file.path(reports_dir, 'fg_success_calibration.csv')
aipw_path <- file.path(reports_dir, 'aipw_calibration.csv')

fg_calibration <- if (file.exists(fg_calibration_path)) {
  readr::read_csv(fg_calibration_path, show_col_types = FALSE)
} else {
  tibble(distance_bin = factor(), n = integer(), rate = numeric(), mean_pred = numeric())
}

aipw_calibration <- if (file.exists(aipw_path)) {
  readr::read_csv(aipw_path, show_col_types = FALSE)
} else {
  tibble(distance_bin = factor(), aipw_mean = numeric())
}

fg_calibration <- fg_calibration %>%
  mutate(mean_pred = dplyr::coalesce(mean_pred, rate))

get_schema(fg_calibration) %>% print(n = 10)


In [ ]:
# Stage Logic — Reporting Views
## TODO: replace placeholder summaries with production reporting pipeline.

predictive_view <- fg_calibration %>%
  mutate(view = 'predictive') %>%
  select(view, distance_bin, n, rate, mean_pred = dplyr::coalesce(mean_pred, rate))

standardized_view <- fg_calibration %>%
  mutate(
    view = 'standardized',
    adjusted_rate = rate + 0.02,
    reference_state = 'Reference grid'
  ) %>%
  select(view, distance_bin, n, adjusted_rate, reference_state)

readr::write_csv(predictive_view, file.path(reports_dir, 'reporting_predictive_view.csv'))
readr::write_csv(standardized_view, file.path(reports_dir, 'reporting_standardized_view.csv'))

comparison_plot <- ggplot(predictive_view, aes(x = distance_bin, y = mean_pred, group = 1)) +
  geom_line(color = '#1b9e77') +
  geom_line(data = standardized_view, aes(y = adjusted_rate, group = 1), color = '#d95f02') +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = 'Predictive vs Standardized Skill', x = 'Distance bin', y = 'Rate / Score')

ggplot2::ggsave(
  filename = file.path(reports_dir, 'reporting_views_comparison.png'),
  plot = comparison_plot,
  width = 7,
  height = 4,
  dpi = 150
)

# Artifact note
message('Stubbed predictive and standardized reporting outputs.')


### Artifacts
- See generated files under `reports/` when the notebook is executed.


In [ ]:
# Session Info
info <- capture.output(sessionInfo())
readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
cat(info, sep = '
')
